In [ ]:
# Required imports
from typing import Any, Dict, List, Optional, Callable, get_type_hints
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import json
import inspect
import re
import math

print("✅ Imports successful")

## 1. What are Tools?

**Tools** are functions that AI agents can call to:
- Perform calculations
- Search the web
- Read/write files
- Query databases
- Call APIs
- Anything a function can do!

The LLM decides **when** to use a tool and **what arguments** to pass.

In [ ]:
# Simple example: A calculator tool

def calculate(expression: str) -> float:
    """
    Evaluate a mathematical expression.
    
    Args:
        expression: A math expression like "2 + 3 * 4"
    
    Returns:
        The result as a float
    """
    # Safe evaluation (in production, use a proper parser)
    allowed = set('0123456789+-*/.() ')
    if not all(c in allowed for c in expression):
        raise ValueError(f"Invalid characters in expression: {expression}")
    
    return float(eval(expression))

# Test it
print(f"calculate('2 + 3 * 4') = {calculate('2 + 3 * 4')}")
print(f"calculate('(10 + 5) / 3') = {calculate('(10 + 5) / 3')}")

## 2. Tool Schema

For an LLM to use a tool, it needs to know:
1. **Name**: What to call it
2. **Description**: What it does
3. **Parameters**: What arguments it accepts

This is described in **JSON Schema** format, compatible with OpenAI's function calling.

In [ ]:
@dataclass
class ToolParameter:
    """Definition of a single tool parameter."""
    name: str
    type: str  # "string", "number", "boolean", "array", "object"
    description: str
    required: bool = True
    default: Any = None
    enum: Optional[List[Any]] = None  # Allowed values
    
    def to_schema(self) -> Dict[str, Any]:
        """Convert to JSON Schema format."""
        schema = {
            "type": self.type,
            "description": self.description
        }
        if self.enum:
            schema["enum"] = self.enum
        if self.default is not None:
            schema["default"] = self.default
        return schema

print("✅ ToolParameter class defined")

In [ ]:
@dataclass
class ToolSchema:
    """Complete schema for a tool."""
    name: str
    description: str
    parameters: List[ToolParameter] = field(default_factory=list)
    
    def to_openai_schema(self) -> Dict[str, Any]:
        """Convert to OpenAI function calling format."""
        properties = {}
        required = []
        
        for param in self.parameters:
            properties[param.name] = param.to_schema()
            if param.required:
                required.append(param.name)
        
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required
                }
            }
        }

print("✅ ToolSchema class defined")

In [ ]:
# Create schema for our calculator tool

calculator_schema = ToolSchema(
    name="calculator",
    description="Evaluate mathematical expressions. Use for arithmetic, percentages, and basic math.",
    parameters=[
        ToolParameter(
            name="expression",
            type="string",
            description="The mathematical expression to evaluate, e.g., '2 + 3 * 4' or '0.15 * 80'",
            required=True
        )
    ]
)

# View the OpenAI-compatible schema
print("OpenAI Function Schema:")
print(json.dumps(calculator_schema.to_openai_schema(), indent=2))

## 3. Tool Implementation

Let's create a proper `Tool` class that combines:
- Schema (for LLM)
- Implementation (for execution)

In [ ]:
@dataclass
class ToolResult:
    """Result from executing a tool."""
    tool_name: str
    tool_input: Dict[str, Any]
    output: Any = None
    success: bool = True
    error: Optional[str] = None
    
    @property
    def output_str(self) -> str:
        """String representation for LLM."""
        if not self.success:
            return f"Error: {self.error}"
        if isinstance(self.output, str):
            return self.output
        return json.dumps(self.output, default=str)

print("✅ ToolResult class defined")

In [ ]:
class BaseTool(ABC):
    """Abstract base class for all tools."""
    
    @property
    @abstractmethod
    def name(self) -> str:
        """Unique name of the tool."""
        pass
    
    @property
    @abstractmethod
    def description(self) -> str:
        """Description of what the tool does."""
        pass
    
    @property
    def parameters(self) -> List[ToolParameter]:
        """Parameters accepted by the tool."""
        return []
    
    @abstractmethod
    def execute(self, **kwargs) -> Any:
        """Execute the tool with given arguments."""
        pass
    
    def run(self, **kwargs) -> ToolResult:
        """Run the tool and return a ToolResult."""
        try:
            output = self.execute(**kwargs)
            return ToolResult(
                tool_name=self.name,
                tool_input=kwargs,
                output=output,
                success=True
            )
        except Exception as e:
            return ToolResult(
                tool_name=self.name,
                tool_input=kwargs,
                output=None,
                success=False,
                error=str(e)
            )
    
    def get_schema(self) -> ToolSchema:
        """Get the tool's schema."""
        return ToolSchema(
            name=self.name,
            description=self.description,
            parameters=self.parameters
        )

print("✅ BaseTool class defined")

In [ ]:
# Implement the Calculator tool

class CalculatorTool(BaseTool):
    """A safe calculator tool."""
    
    @property
    def name(self) -> str:
        return "calculator"
    
    @property
    def description(self) -> str:
        return "Evaluate mathematical expressions. Supports +, -, *, /, parentheses, and decimal numbers."
    
    @property
    def parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="expression",
                type="string",
                description="Math expression like '2 + 3 * 4' or '(10 + 5) / 3'",
                required=True
            )
        ]
    
    def execute(self, expression: str) -> float:
        """Safely evaluate a math expression."""
        # Validate characters
        allowed = set('0123456789+-*/.() ')
        if not all(c in allowed for c in expression):
            raise ValueError(f"Invalid characters in expression")
        
        # Evaluate
        result = eval(expression)
        return float(result)

# Test the tool
calc = CalculatorTool()

print(f"Tool name: {calc.name}")
print(f"Tool description: {calc.description}")
print()

# Run some calculations
result1 = calc.run(expression="15 * 80 / 100")
print(f"15% of 80: {result1.output_str}")

result2 = calc.run(expression="(25 + 75) * 2")
print(f"(25 + 75) * 2: {result2.output_str}")

# Test error handling
result3 = calc.run(expression="10 / 0")
print(f"10 / 0: {result3.output_str}")

## 4. Tool Registry

A **Tool Registry** manages multiple tools:
- Register tools by name
- Look up tools
- Get all schemas for LLM
- Execute tools by name

In [ ]:
class ToolRegistry:
    """Registry for managing multiple tools."""
    
    def __init__(self):
        self._tools: Dict[str, BaseTool] = {}
    
    def register(self, tool: BaseTool) -> None:
        """Register a tool."""
        if tool.name in self._tools:
            raise ValueError(f"Tool '{tool.name}' already registered")
        self._tools[tool.name] = tool
        print(f"✅ Registered tool: {tool.name}")
    
    def get(self, name: str) -> Optional[BaseTool]:
        """Get a tool by name."""
        return self._tools.get(name)
    
    def list_tools(self) -> List[str]:
        """List all registered tool names."""
        return list(self._tools.keys())
    
    def get_all_schemas(self) -> List[Dict[str, Any]]:
        """Get OpenAI schemas for all tools."""
        return [
            tool.get_schema().to_openai_schema()
            for tool in self._tools.values()
        ]
    
    def execute(self, tool_name: str, **kwargs) -> ToolResult:
        """Execute a tool by name."""
        tool = self.get(tool_name)
        if not tool:
            return ToolResult(
                tool_name=tool_name,
                tool_input=kwargs,
                success=False,
                error=f"Tool '{tool_name}' not found"
            )
        return tool.run(**kwargs)

print("✅ ToolRegistry class defined")

In [ ]:
# Create more tools

class TextProcessorTool(BaseTool):
    """Process and manipulate text."""
    
    @property
    def name(self) -> str:
        return "text_processor"
    
    @property
    def description(self) -> str:
        return "Process text: count words, characters, extract patterns, or transform text."
    
    @property
    def parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="text",
                type="string",
                description="The text to process",
                required=True
            ),
            ToolParameter(
                name="operation",
                type="string",
                description="Operation to perform",
                required=True,
                enum=["word_count", "char_count", "uppercase", "lowercase", "reverse"]
            )
        ]
    
    def execute(self, text: str, operation: str) -> Any:
        operations = {
            "word_count": lambda t: len(t.split()),
            "char_count": lambda t: len(t),
            "uppercase": lambda t: t.upper(),
            "lowercase": lambda t: t.lower(),
            "reverse": lambda t: t[::-1]
        }
        
        if operation not in operations:
            raise ValueError(f"Unknown operation: {operation}")
        
        return operations[operation](text)


class DateTimeTool(BaseTool):
    """Get current date and time."""
    
    @property
    def name(self) -> str:
        return "datetime"
    
    @property
    def description(self) -> str:
        return "Get current date, time, or calculate date differences."
    
    @property
    def parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="query",
                type="string",
                description="What to get: 'now', 'date', 'time', 'weekday', 'year'",
                required=True
            )
        ]
    
    def execute(self, query: str) -> str:
        from datetime import datetime
        now = datetime.now()
        
        queries = {
            "now": now.isoformat(),
            "date": now.strftime("%Y-%m-%d"),
            "time": now.strftime("%H:%M:%S"),
            "weekday": now.strftime("%A"),
            "year": str(now.year)
        }
        
        if query.lower() not in queries:
            raise ValueError(f"Unknown query: {query}")
        
        return queries[query.lower()]

print("✅ Additional tools defined")

In [ ]:
# Create a registry and add tools

registry = ToolRegistry()
registry.register(CalculatorTool())
registry.register(TextProcessorTool())
registry.register(DateTimeTool())

print(f"\nRegistered tools: {registry.list_tools()}")

In [ ]:
# Execute tools through the registry

print("🔧 Tool Executions:")
print("=" * 50)

# Calculator
result = registry.execute("calculator", expression="100 * 1.15")
print(f"calculator('100 * 1.15') = {result.output_str}")

# Text processor
result = registry.execute("text_processor", text="Hello World", operation="word_count")
print(f"text_processor('Hello World', 'word_count') = {result.output_str}")

result = registry.execute("text_processor", text="Hello World", operation="uppercase")
print(f"text_processor('Hello World', 'uppercase') = {result.output_str}")

# DateTime
result = registry.execute("datetime", query="weekday")
print(f"datetime('weekday') = {result.output_str}")

# Non-existent tool
result = registry.execute("fake_tool", arg="test")
print(f"fake_tool('test') = {result.output_str}")

In [ ]:
# Get all schemas (this would be sent to the LLM)

print("📋 All Tool Schemas for LLM:")
print("=" * 50)

schemas = registry.get_all_schemas()
print(json.dumps(schemas, indent=2))

## 5. Tool Decorators

Creating tool classes is verbose. Let's use **decorators** to make it elegant!

In [ ]:
def _python_type_to_json_type(python_type: type) -> str:
    """Convert Python type to JSON schema type."""
    type_map = {
        str: "string",
        int: "number",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
    }
    return type_map.get(python_type, "string")


def _parse_docstring(docstring: str) -> Dict[str, str]:
    """Parse docstring to extract parameter descriptions."""
    if not docstring:
        return {}
    
    params = {}
    lines = docstring.split('\n')
    current_param = None
    
    for line in lines:
        line = line.strip()
        # Match "param_name: description" or "param_name (type): description"
        match = re.match(r'^(\w+)(?:\s*\([^)]+\))?:\s*(.+)$', line)
        if match:
            current_param = match.group(1)
            params[current_param] = match.group(2)
        elif current_param and line and not line.startswith('Args') and not line.startswith('Returns'):
            # Continue previous parameter description
            if current_param in params:
                params[current_param] += ' ' + line
    
    return params

print("✅ Helper functions defined")

In [ ]:
class FunctionTool(BaseTool):
    """Tool wrapper for a regular function."""
    
    def __init__(self, func: Callable, name: str = None, description: str = None):
        self._func = func
        self._name = name or func.__name__
        self._description = description or func.__doc__ or "No description"
        self._parameters = self._extract_parameters()
    
    def _extract_parameters(self) -> List[ToolParameter]:
        """Extract parameters from function signature."""
        sig = inspect.signature(self._func)
        hints = get_type_hints(self._func) if hasattr(self._func, '__annotations__') else {}
        doc_params = _parse_docstring(self._func.__doc__ or "")
        
        params = []
        for name, param in sig.parameters.items():
            if name in ('self', 'cls'):
                continue
            
            python_type = hints.get(name, str)
            json_type = _python_type_to_json_type(python_type)
            description = doc_params.get(name, f"Parameter: {name}")
            required = param.default == inspect.Parameter.empty
            default = None if required else param.default
            
            params.append(ToolParameter(
                name=name,
                type=json_type,
                description=description,
                required=required,
                default=default
            ))
        
        return params
    
    @property
    def name(self) -> str:
        return self._name
    
    @property
    def description(self) -> str:
        # Get first line of description
        return self._description.split('\n')[0].strip()
    
    @property
    def parameters(self) -> List[ToolParameter]:
        return self._parameters
    
    def execute(self, **kwargs) -> Any:
        return self._func(**kwargs)

print("✅ FunctionTool class defined")

In [ ]:
def tool(name: str = None, description: str = None):
    """
    Decorator to convert a function into a Tool.
    
    Usage:
        @tool(name="my_tool", description="Does something")
        def my_function(arg1: str, arg2: int = 10) -> str:
            ...
    """
    def decorator(func: Callable) -> FunctionTool:
        return FunctionTool(func, name=name, description=description)
    return decorator

print("✅ @tool decorator defined")

In [ ]:
# Create tools using the decorator

@tool(name="search", description="Search for information on a topic")
def search_web(query: str, max_results: int = 5) -> str:
    """
    Search the web for information.
    
    Args:
        query: The search query
        max_results: Maximum number of results to return
    
    Returns:
        Search results as a string
    """
    # Simulated search results
    return f"Found {max_results} results for '{query}': [simulated results]"


@tool()
def get_weather(city: str, units: str = "celsius") -> Dict[str, Any]:
    """
    Get the current weather for a city.
    
    Args:
        city: The city name
        units: Temperature units (celsius or fahrenheit)
    """
    # Simulated weather
    import random
    temp = random.randint(15, 30) if units == "celsius" else random.randint(59, 86)
    return {
        "city": city,
        "temperature": temp,
        "units": units,
        "condition": random.choice(["sunny", "cloudy", "rainy"])
    }


@tool(name="math_sqrt")
def square_root(number: float) -> float:
    """Calculate the square root of a number."""
    if number < 0:
        raise ValueError("Cannot calculate square root of negative number")
    return math.sqrt(number)

print("✅ Decorator-based tools created")

In [ ]:
# Test the decorated tools

print(f"Tool: {search_web.name}")
print(f"Description: {search_web.description}")
print(f"Parameters: {[p.name for p in search_web.parameters]}")
print()

# Execute
result = search_web.run(query="Python tutorials")
print(f"Result: {result.output_str}")
print()

result = get_weather.run(city="London")
print(f"Weather: {result.output_str}")
print()

result = square_root.run(number=16)
print(f"sqrt(16) = {result.output_str}")

In [ ]:
# Add to registry

registry2 = ToolRegistry()
registry2.register(search_web)
registry2.register(get_weather)
registry2.register(square_root)

# View schemas
print("\n📋 Schemas:")
for schema in registry2.get_all_schemas():
    print(f"  - {schema['function']['name']}: {schema['function']['description']}")

## 6. Simulating LLM Tool Calls

Let's see how this would work with an LLM:

In [ ]:
def simulate_llm_tool_use(user_query: str, registry: ToolRegistry) -> str:
    """
    Simulate how an LLM would use tools.
    In practice, this would be the actual LLM API call.
    """
    print(f"\n🧠 Processing: {user_query}")
    print("=" * 50)
    
    # Simulate LLM deciding to use a tool
    # (In reality, this comes from the LLM's response)
    
    if "weather" in user_query.lower():
        # LLM chooses get_weather tool
        city = "London"  # Would be extracted by LLM
        print(f"📝 LLM decides to use 'get_weather' with city='{city}'")
        
        result = registry.execute("get_weather", city=city)
        print(f"🔧 Tool result: {result.output_str}")
        
        # LLM formulates response using tool output
        response = f"The weather in {city} is {result.output['condition']} with a temperature of {result.output['temperature']}°C."
        
    elif "sqrt" in user_query.lower() or "square root" in user_query.lower():
        # LLM chooses math_sqrt tool
        number = 144  # Would be extracted by LLM
        print(f"📝 LLM decides to use 'math_sqrt' with number={number}")
        
        result = registry.execute("math_sqrt", number=number)
        print(f"🔧 Tool result: {result.output_str}")
        
        response = f"The square root of {number} is {result.output}."
        
    else:
        # LLM uses search tool
        print(f"📝 LLM decides to use 'search' with query='{user_query}'")
        
        result = registry.execute("search", query=user_query)
        print(f"🔧 Tool result: {result.output_str}")
        
        response = f"Based on my search: {result.output_str}"
    
    print(f"\n✅ Final Response: {response}")
    return response

# Simulate some queries
simulate_llm_tool_use("What's the weather in London?", registry2)
simulate_llm_tool_use("What's the square root of 144?", registry2)
simulate_llm_tool_use("Tell me about machine learning", registry2)

## Summary

In this notebook, we learned how to build tools for AI agents:

| Component | Purpose |
|-----------|--------|
| `ToolParameter` | Describes a single parameter (name, type, description) |
| `ToolSchema` | Complete tool definition with all parameters |
| `BaseTool` | Abstract base class for implementing tools |
| `ToolResult` | Captures execution output (success/failure) |
| `ToolRegistry` | Manages multiple tools, provides schemas |
| `@tool` decorator | Easy way to create tools from functions |

### Key Concepts:
1. **Tools are functions** that LLMs can call
2. **Schemas describe tools** in JSON format for the LLM
3. **Registry manages tools** and provides execution
4. **Decorators make it easy** to create tools from functions
5. **Results are structured** with success/failure handling